In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()


In [0]:
# Config: single source of truth (replaces DataLakeConfig import)
%run ../../config/pipeline_config

In [0]:
# Load audit logger for error-only logging
%run ../../utils/audit_logger

In [0]:
import sys
import os
from datetime import datetime
from pyspark.sql.functions import max as spark_max

# 1. Widgets & Setup
dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("bq_native_table", "v4c-bigquery.v4c_bigquery_dataset.event_raw")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bq_native_table = dbutils.widgets.get("bq_native_table")
load_type = "incremental"

# --------------------------------------------------
# 2. Import PipelineMetrics (config loaded via %run in previous cell)
# --------------------------------------------------
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
abs_project_root = os.path.abspath(project_root)

if abs_project_root not in sys.path:
    sys.path.append(abs_project_root)

from utils.pipeline_metrics import PipelineMetrics

# ── Initialize metrics ──
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "BQ_Incremental_Load",
    task_key       = "run_job_A_bq_ingestion",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

# Ensure audit tables exist for error logging
initialize_audit_tables()

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 1)

# Generate landing path via pipeline_config helper + timestamp
now = datetime.now()
incremental_path = f"{landing_path(source_system, customer_id, object_name, load_type)}{now.strftime('%Y/%m/%d/%H%M%S')}/"

print(f"🚀 Starting FAST NATIVE Incremental Load to: {incremental_path}")

gcp_project_id = bq_native_table.split(".")[0]

# ── Core pipeline logic with error-only audit logging ──
t_start = datetime.now()

try:
    # 3. Securely Retrieve GCP Credentials
    gcp_secret = dbutils.secrets.get(scope="gcp_auth", key="bq_key")

    # 4. Get the last watermark (exact max_ts from previous run)
    watermark_df = spark.sql(f"""
        SELECT last_processed_timestamp
        FROM ingestion_metadata.watermarks
        WHERE source_system='{source_system}'
        AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing in ingestion_metadata.watermarks. Run historical load first.")
    
    watermark_dt = rows[0][0] 
    # Format the Python datetime into BigQuery's string format (ISO 8601)
    watermark_str = watermark_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    
    print(f"📍 Last Watermark (exact): {watermark_str}")

    # 5. Query BigQuery for the NEW max timestamp
    # Uses strict > so the record at exactly watermark_str is excluded (already loaded)
    new_ts_df = spark.read \
        .format("bigquery") \
        .option("credentials", gcp_secret) \
        .option("parentProject", gcp_project_id) \
        .option("project", gcp_project_id) \
        .option("filter", f"event_timestamp > '{watermark_str}'") \
        .load(bq_native_table) \
        .select(spark_max("event_timestamp").alias("max_ts"))

    new_ts_str = new_ts_df.first()[0]

    if new_ts_str is None:
        print("✅ No new records found in BigQuery. Exiting gracefully.")
        dbutils.notebook.exit("0")

    print(f"🎯 New records found up to: {new_ts_str}")

    # 6. Pull the data using Native Connector
    # Strict > on watermark excludes last record of previous run → zero duplicates
    df_incremental = spark.read \
        .format("bigquery") \
        .option("credentials", gcp_secret) \
        .option("parentProject", gcp_project_id) \
        .option("project", gcp_project_id) \
        .option("filter", f"event_timestamp > '{watermark_str}' AND event_timestamp <= '{new_ts_str}'") \
        .load(bq_native_table)

    # ✅ FIX H5: Count BEFORE write (Serverless-compatible, no cache needed)
    total_rows = df_incremental.count()
    print(f"📊 Processing {total_rows:,} rows")

    # ✅ FIX M8: Increased from repartition(2) to repartition(10) for better file sizes
    df_incremental.repartition(10).write \
        .mode("append") \
        .format("parquet") \
        .option("compression", "snappy") \
        .save(incremental_path)
    print("💾 Write to S3 completed successfully.")

    # 7. Update Watermark Table — store exact max_ts (no lookback)
    # Next run uses strict > so this exact timestamp is excluded → zero duplicates, zero gaps
    print("Updating watermark table with exact max timestamp...")

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW new_watermarks AS
        SELECT source_system, object_name, 
               CASE 
                   WHEN source_system = '{source_system}' AND object_name = '{object_name}' 
                   THEN CAST('{new_ts_str}' AS TIMESTAMP)
                   ELSE last_processed_timestamp 
               END as last_processed_timestamp
        FROM ingestion_metadata.watermarks
    """)

    spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")

    print(f"✅ Watermark updated to exact max_ts: {new_ts_str}")

    # Set task value for downstream orchestration
    dbutils.jobs.taskValues.set(key="bq_files_count", value=str(total_rows))
    pm.save(status="SUCCESS", rows_processed=total_rows)

except Exception as e:
    elapsed_ms = int((datetime.now() - t_start).total_seconds() * 1000)
    print(f"❌ Pipeline Failed: {e}")
    log_audit(
        job_name       = "BQ_Incremental_Load",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "bronze",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
        duration_ms    = elapsed_ms,
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise e